# Explore the spatial correlation, using spin test to determine the significance level
## Xiao Chen
## chenxiaophd@gmail.com

In [4]:
import numpy as np
import pandas as pd
import neuromaps
from neuromaps import stats
from neuromaps import datasets, images, nulls, resampling
from neuromaps.datasets import available_annotations
from neuromaps.datasets import fetch_annotation
from neuromaps.resampling import resample_images
import os
import matplotlib.pyplot as plt
import seaborn as sns
import nibabel as nib

## The stable results
## 260625

## legacy code: get the gradient maps

In [ ]:
# Gradient 1
g1 = fetch_annotation(
    source="margulies2016",
    desc="fcgradient01",
    space="fsLR",
    den="32k"
)

# Gradient 2
g2 = fetch_annotation(
    source="margulies2016",
    desc="fcgradient02",
    space="fsLR",
    den="32k"
)

# Gradient 3
g3 = fetch_annotation(
    source="margulies2016",
    desc="fcgradient03",
    space="fsLR",
    den="32k"
)

print("Gradient 1:", g1)
print("Gradient 2:", g2)
print("Gradient 3:", g3)

## set the paths

In [5]:
work_dir = "xxxxxxxx"
output_dir = os.path.join(work_dir, "xxxxxxxx/spatial_corr")
map_dir = "~/rumination_lesion_network_mapping/maps"
atlas_dir = "xxxxxx/atlas"

## prepare all the annotations

In [6]:
scale_set = ["rumination"]
group_set = ["MDD"]
GSR_set = ["_GSR"]

# t maps
causal_mapping = []
causal_mapping_flag = []
for scale in scale_set:
    for group in group_set:
        for GSR in GSR_set:
            temp_lh = os.path.join(map_dir, f"{scale}_{group}{GSR}_lh.gii")
            temp_rh = os.path.join(map_dir, f"{scale}_{group}{GSR}_rh.gii")
            causal_mapping.append([temp_lh, temp_rh])
            causal_mapping_flag.append(f"{scale}_{group}{GSR}")

# interested maps
anno_interested = []
anno_intereste_flag = []
map_interested_set = ["margulies2016_fcgradient01", "margulies2016_fcgradient02", "margulies2016_fcgradient03"]

for map_interested in map_interested_set:
    temp_lh = os.path.join(atlas_dir, f"{map_interested}_fsLR_32k_lh_fsaverage10k.func.gii")
    temp_rh = os.path.join(atlas_dir, f"{map_interested}_fsLR_32k_rh_fsaverage10k.func.gii")
    anno_interested.append([temp_lh, temp_rh])
    anno_intereste_flag.append(map_interested)

## do correction and calculate the spatial corr

In [7]:
from scipy.stats import spearmanr


def gradient_spearman_metric(source, target):
    """Calculate Spearman's rho for each observed or rotated map pair."""
    # Rank each pair after masking, including every supplied spin null map.
    # The neuromaps metric="spearmanr" shortcut can leave null maps unranked.
    if len(source) < 2 or np.ptp(source) == 0 or np.ptp(target) == 0:
        return np.float64(np.nan)
    return np.float64(spearmanr(source, target).statistic)


# Bonferroni correction across the three gradients.
alpha = 0.05/3

n_causal_mapping = len(causal_mapping)
n_anno = len(map_interested_set)

r_mat = np.zeros((n_causal_mapping, n_anno))
p_mat = np.zeros((n_causal_mapping, n_anno))
rows = []
for i, c_name in enumerate(causal_mapping):
    rotated = neuromaps.nulls.alexander_bloch(c_name, atlas='fsaverage', density='10k', 
                                    n_perm=5000, seed=1234)
    for j, a_name in enumerate(anno_interested):
        corr, pval, null_corr = stats.compare_images(
            c_name, a_name, metric=gradient_spearman_metric,
            nulls=rotated, return_nulls=True,
        )
        if not np.isfinite(corr) or not np.isfinite(pval) or not np.isfinite(null_corr).all():
            raise ValueError(f"Undefined observed/null Spearman correlation: {c_name} vs {a_name}")
        r_mat[i, j] = corr
        p_mat[i, j] = pval
        
        rows.append({
            "causal_map": causal_mapping_flag[i],
            "anno_interested": anno_intereste_flag[j],
            "r": corr,  # Spearman's rho; retain the CSV column for compatibility.
            "p": pval,
            "significant_(p<0.05)": pval < alpha
        })

## save out results
output_path = os.path.join(output_dir, "spatial_corr.csv")
df = pd.DataFrame(rows)
df = df.rename(
    columns={
        "anno_interested": "annotation",
        "significant_(p<0.05)": "significant_p05",
    }
)

# Match the annotation naming convention in the reference CSV.
df["annotation"] = df["annotation"].apply(
    lambda value: (
        value
        if value.endswith("_fsLR_32k")
        else f"{value}_fsLR_32k"
    )
)

# Add an asterisk for significant correlations.
df["sig_label"] = df["significant_p05"].map(
    {True: "**", False: ""}
)

# Match the column order in the reference CSV.
df = df[
    [
        "causal_map",
        "annotation",
        "r",
        "p",
        "significant_p05",
        "sig_label",
    ]
]

df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: /Users/Chenxiao/Documents/project_codes/rumination_casual_mapping/analyses/stable_260725/spatial_corr/spatial_corr.csv


## Bin whole-brain vertices by Gradient 3 and summarize LNM

Gradient 3 is first resampled to the LNM surface space and density with `neuromaps`. Vertices where both maps equal zero are removed, then the left and right hemispheres are concatenated before calculating whole-brain deciles. The output columns match the reference long and summary CSV files.

In [ ]:
# Use the Gradient 3 annotation and the LNM map prepared above.
gradient3_paths = anno_interested[2]
lnm_paths = causal_mapping[0]

# Align Gradient 3 to the LNM surface space and vertex density before
# extracting values. The returned LNM images are kept in the same target grid.
gradient3_resampled, lnm_resampled = resample_images(
    src=gradient3_paths,
    trg=lnm_paths,
    src_space="fsaverage",
    trg_space="fsaverage",
    method="linear",
    resampling="transform_to_trg",
)

vertex_tables = []
for hemi, gradient_img, lnm_img in zip(
    ["L", "R"], gradient3_resampled, lnm_resampled
):
    gradient = np.asarray(gradient_img.agg_data()).squeeze()
    lnm = np.asarray(lnm_img.agg_data()).squeeze()

    if gradient.ndim != 1 or lnm.ndim != 1:
        raise ValueError(f"Expected one-dimensional surface data for hemisphere {hemi}.")
    if gradient.shape != lnm.shape:
        raise ValueError(
            f"Gradient 3 and LNM vertex counts differ for hemisphere {hemi}: "
            f"{gradient.size} versus {lnm.size}."
        )

    vertex_tables.append(
        pd.DataFrame(
            {
                "gradient": gradient,
                "value": lnm,
                "hemi": hemi,
            }
        )
    )

df_bins = pd.concat(vertex_tables, ignore_index=True)

if not np.isfinite(df_bins[["gradient", "value"]].to_numpy()).all():
    raise ValueError("Gradient 3 or LNM contains NaN/Inf values; resolve them before binning.")

# Remove medial-wall/background rows only when both aligned maps equal zero.
both_zero = df_bins["gradient"].eq(0) & df_bins["value"].eq(0)
n_both_zero_removed = int(both_zero.sum())
df_bins = df_bins.loc[~both_zero].reset_index(drop=True)

# Calculate deciles once across both hemispheres. Tied gradient values stay
# together, so bin sizes can differ slightly at a percentile boundary.
df_bins["bin"] = pd.qcut(
    df_bins["gradient"],
    q=10,
    labels=range(1, 11),
).astype(int)

summary_df = (
    df_bins.groupby("bin", as_index=False, observed=True)["value"]
    .agg(mean="mean", sd="std", n="size")
)
summary_df["sem"] = summary_df["sd"] / np.sqrt(summary_df["n"])
summary_df["label"] = [f"{lower}-{lower + 10}%" for lower in range(0, 100, 10)]

expected_bins = list(range(1, 11))
if summary_df["bin"].tolist() != expected_bins:
    raise RuntimeError(f"Expected bins {expected_bins}, got {summary_df['bin'].tolist()}.")
if not summary_df["mean"].notna().all():
    raise RuntimeError("At least one Gradient 3 bin has no LNM values.")

os.makedirs(output_dir, exist_ok=True)
long_output_path = os.path.join(output_dir, "LNM_gradient_binned_long.csv")
summary_output_path = os.path.join(output_dir, "LNM_gradient_binned_summary.csv")

df_bins[["gradient", "value", "hemi", "bin"]].to_csv(
    long_output_path, index=False
)
summary_df[["bin", "mean", "sd", "n", "sem", "label"]].to_csv(
    summary_output_path, index=False
)

print(f"Removed {n_both_zero_removed} vertices where Gradient 3 and LNM both equal zero.")
print(summary_df.to_string(index=False))
print(f"Saved long-format vertex data to: {long_output_path}")
print(f"Saved bin summary to: {summary_output_path}")